In [1]:
import tensorflow as tf
from tensorflow.keras import layers
import keras

In [2]:
class ResBlock(layers.Layer):
    def __init__(self, filters, stride=1):
        super().__init__()
        self.filters = filters
        self.stride = stride
        
        self.conv1 = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False, kernel_initializer='he_normal')
        self.bn1 = layers.BatchNormalization()
        self.act1 = layers.ReLU()
        
        self.conv2 = layers.Conv2D(filters, 3, padding='same', use_bias=False, kernel_initializer='he_normal')
        self.bn2 = layers.BatchNormalization()
        
        self.proj = None
        self.proj_bn = None
        self.out_act = layers.ReLU()
        
    def build(self, input_shape):
        in_ch = input_shape[-1]
        
        if self.stride != 1 or in_ch != self.filters:
            self.proj = layers.Conv2D(self.filters, 1, strides=self.stride, padding='same', use_bias=False, kernel_initializer='he_normal')
            self.proj_bn = layers.BatchNormalization()
        
        super().build(input_shape)
        
    def call(self, x, training=False):
        shortcut = x
        
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = self.act1(x)
        
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        
        if self.proj is not None:
            shortcut = self.proj(shortcut)
            shortcut = self.proj_bn(shortcut, training=training)
        
        x = x + shortcut
        x = self.out_act(x)
        return x

In [3]:
class BackboneFPN(layers.Layer):
    def __init__(self, fpn_channels=256):
        super().__init__()
        self.stem_conv = layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False, kernel_initializer='he_normal')
        self.stem_bn = layers.BatchNormalization()
        self.stem_act = layers.ReLU()
        self.stem_pool = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')
        
        self.r2_1 = ResBlock(64, stride=1)
        self.r2_2 = ResBlock(64, stride=1)
        
        self.r3_1 = ResBlock(128, stride=2)
        self.r3_2 = ResBlock(128, stride=1)
        
        self.r4_1 = ResBlock(256, stride=2)
        self.r4_2 = ResBlock(256, stride=1)
        
        self.r5_1 = ResBlock(512, stride=2)
        self.r5_2 = ResBlock(512, stride=2)
        
        self.c3_lat = layers.Conv2D(fpn_channels, 1, padding='same')
        self.c4_lat = layers.Conv2D(fpn_channels, 1, padding='same')
        self.c5_lat = layers.Conv2D(fpn_channels, 1, padding='same')
        
        self.p3_out = layers.Conv2D(fpn_channels, 3, padding='same')
        self.p4_out = layers.Conv2D(fpn_channels, 3, padding='same')
        self.p5_out = layers.Conv2D(fpn_channels, 3, padding='same')
        
        self.p6_pool = layers.MaxPooling2D(pool_size=1, strides=2)
        
        self.up2x = layers.UpSampling2D(size=(2, 2), interpolation='nearest')
    
    def call(self, images, training=False):
        x = self.stem_conv(images)
        x = self.stem_bn(x, training=training)
        x = self.stem_act(x)
        x = self.stem_pool(x)
        
        x = self.r2_1(x, training=training)
        x = self.r2_2(x, training=training)
        
        c3 = self.r3_1(x, training=training)
        c3 = self.r3_2(c3, training=training)
        
        c4 = self.r4_1(c3, training=training)
        c4 = self.r4_2(c4, training=training)

        c5 = self.r5_1(c4, training=training)
        c5 = self.r5_2(c5, training=training)
        
        p5 = self.c5_lat(c5)
        p4 = self.c4_lat(c4) + self.up2x(p5)
        p3 = self.c3_lat(c3) + self.up2x(p4)
        
        p3 = self.p3_out(p3)
        p4 = self.p3_out(p4)
        p5 = self.p3_out(p5)
        p6 = self.p3_out(p6)
        
        return {"p3": p3, "p4": p4, "p5": p5, "p6": p6}

In [4]:
class ROIAlign(layers.Layer):
    def __init__(self, output_size):
        super().__init__()
        self.output_size = output_size
        
    def call(self, feature_map, rois, roi_batch_indices):
        return tf.image.crop_and_resize(
            image=feature_map,
            boxes=rois,
            box_indices=roi_batch_indices,
            crop_size=self.output_size,
            method='bilinear'
        )

In [5]:
class BoxHead(layers.Layer):
    def __init__(self, num_classes, fc_units=1024):
        super().__init__()
        self.flatten = layers.Flatten()
        self.fc1 = layers.Dense(fc_units, activation='relu')
        self.fc2 = layers.Dense(fc_units, activation='relu')
        self.cls_logits = layers.Dense(num_classes)
        self.bbox_deltas = layers.Dense(4)
        
    def call(self, roi_feats):
        x = self.flatten(roi_feats)
        x = self.fc1(x)
        x = self.fc2(x)
        cls_logits = self.cls_logits(x)
        bbox_deltas = self.bbox_deltas(x)
        return cls_logits, bbox_deltas

In [6]:
class MaskHead(layers.Layer):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = layers.Conv2D(256, 3, padding='same', activation='relu')
        self.conv2 = layers.Conv2D(256, 3, padding='same', activation='relu')
        self.conv3 = layers.Conv2D(256, 3, padding='same', activation='relu')
        self.conv4 = layers.Conv2D(256, 3, padding='same', activation='relu')
        self.deconv = layers.Conv2DTranspose(256, 2, strides=2, activation='relu')
        self.mask_logits = layers.Conv2D(num_classes, 1, padding='same')
    
    def call(self, roi_feats):
        x = self.conv1(roi_feats)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.deconv(x)
        x = self.mask_logits(x)
        return x

#### Simplified Mask R-CNN (Stage-2)

In [8]:
class SimplifiedMaskRCNN(tf.keras.Model):
    def __init__(self, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.backbone_fpn = BackboneFPN(fpn_channels=256)
        self.roi_align_box = ROIAlign((7, 7))
        self.roi_align_mask = ROIAlign((14, 14))
        self.box_head = BoxHead(num_classes=num_classes)
        self.mask_head = MaskHead(num_classes=num_classes)
    
    def call(self, inputs, training=False):
        images = inputs["images"]
        rois = inputs["rois"]
        roi_batch_indices = inputs["roi_batch_indices"]
        
        fpn_feats = self.backbone_fpn(images, training=training)
        
        p3 = fpn_feats["p3"]
        
        roi_box_feats = self.roi_align_box(p3, rois, roi_batch_indices)
        roi_mask_feats = self.roi_align_mask(p3, rois, roi_batch_indices)
        
        cls_logits, bbox_deltas = self.box_head(roi_box_feats)
        mask_logits = self.mask_head(roi_mask_feats)
        
        return {
            "cls_logits": cls_logits,
            "bbox_deltas": bbox_deltas,
            "mask_logits": mask_logits
        }